# Módulo 16 · Semana 16, Pipeline reproducible y data leakage
Juan Esteban Camacho Castro, Juan Diego Serpa Ruiz

## Abrir el notebook y cargar el dataset de features seleccionadas

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import recall_score
import joblib

In [3]:
df = pd.read_csv("novamarket_features_seleccionadas.csv")

## Separar las columnas en dos listas según su naturaleza

In [4]:
for col in df.columns: print(col, df[col].unique()[:5])

nivel_lealtad_cod [1. 0. 3. 2.]
canal_App [1 0]
es_venta_corporativa [False  True]
edad_cliente [68 62 26 73 53]
gasto_promedio [ 572720.         1126450.          197000.          300100.
  344533.33333333]
precio_unitario [432300. 582200. 234200. 140800. 526600.]
monto_compra [432300. 582200. 234200.      0. 526600.]
dias_desde_ultima_compra [636. 662. 659. 596. 663.]
fuga_cliente [1 0]
id_cliente ['C100866' 'C83600' 'C33012' 'C62238' 'C56457']


In [5]:
columnas_continuas = [
    'nivel_lealtad_cod',
    'edad_cliente',
    'gasto_promedio',
    'precio_unitario',
    'monto_compra',
    'dias_desde_ultima_compra'
]

In [6]:
columnas_binarias = [
    'canal_App',
    'es_venta_corporativa'
]

In [7]:
len(columnas_continuas) + len(columnas_binarias)

8

In [ ]:
len(df.columns)

## Construir el sub-pipeline para las columnas continuas

In [11]:
pipeline_continuas = Pipeline(steps=[("imputador", SimpleImputer(strategy="median")), ("escalador", StandardScaler())])

In [12]:
pipeline_continuas.named_steps

{'imputador': SimpleImputer(strategy='median'), 'escalador': StandardScaler()}

## Ensamblar el ColumnTransformer completo

In [14]:
preprocesador = ColumnTransformer(transformers=[("continuas", pipeline_continuas, columnas_continuas), ("binarias", "passthrough", columnas_binarias)])

## Probar el ColumnTransformer con una muestra pequeña

In [15]:
X = df.drop(columns=["fuga_cliente", "id_cliente"])

In [16]:
X_transformado_prueba = preprocesador.fit_transform(X.head(20))

In [17]:
X_transformado_prueba.shape[1]

8

In [18]:
len(columnas_continuas) + len(columnas_binarias)

8

## Envolver el ColumnTransformer dentro de un Pipeline general

In [ ]:
pipeline_novamarket = Pipeline(steps=[("preprocesador", preprocesador), ("modelo_validacion", LogisticRegression(max_iter=1000, class_weight="balanced"))])

In [ ]:
pipeline_novamarket.named_steps

## Separar entrenamiento y prueba, con la variable objetivo

In [ ]:
X = df.drop(columns=["fuga_cliente", "id_cliente"])
y = df["fuga_cliente"]

In [ ]:
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
print("Proporción en entrenamiento:")
print(y_entrenamiento.value_counts(normalize=True))
print("\nProporción en prueba:")
print(y_prueba.value_counts(normalize=True))

## Ejecutar la validación cruzada sobre el Pipeline completo

In [ ]:
resultados = cross_val_score(pipeline_novamarket, X_entrenamiento, y_entrenamiento, cv=5, scoring="recall")

In [ ]:
resultados

## Interpretar el resultado de la validación cruzada

In [ ]:
print("Recall promedio:", resultados.mean())
print("Desviación estándar:", resultados.std())

El `recall` promedio de la validación cruzada mide, en promedio, qué porcentaje de los clientes que sí se fugan logra detectar el pipeline en las 5 rondas. Un valor alto significa que el modelo está capturando bien a la clase minoritaria (los clientes en fuga), que es justamente la que importa para el negocio. La desviación estándar muestra qué tan estable es ese resultado entre las distintas rondas: si es baja, el pipeline funciona de forma consistente; si es alta, el desempeño depende mucho de qué parte de los datos entra en cada fold.

## Entrenar el Pipeline final con todo el entrenamiento

In [ ]:
pipeline_novamarket.fit(X_entrenamiento, y_entrenamiento)

In [ ]:
pipeline_novamarket.named_steps["preprocesador"]

## Confirmar que el conjunto de prueba nunca participó en el ajuste

El pipeline se ajustó (`.fit()`) únicamente con `X_entrenamiento` y `y_entrenamiento` en la celda anterior. El conjunto de prueba (`X_prueba`, `y_prueba`) nunca entró al `fit`, así que no hubo fuga de datos. A continuación observamos el `recall` sobre la prueba solo para comparar con el promedio de la validación cruzada: si sale cercano, el pipeline generaliza bien; si sale muy por encima, sospecharíamos de fuga.

In [ ]:
recall_score(y_prueba, pipeline_novamarket.predict(X_prueba))

## Guardar el pipeline entrenado con joblib

In [ ]:
joblib.dump(pipeline_novamarket, "novamarket_pipeline_v1.joblib")

## Confirmar que el archivo guardado funciona igual que el original

In [ ]:
pipeline_cargado = joblib.load("novamarket_pipeline_v1.joblib")

In [ ]:
recall_score(y_prueba, pipeline_cargado.predict(X_prueba))

El `recall` del pipeline cargado debe ser exactamente igual al del pipeline original. Si coincide, el archivo `.joblib` conserva todo lo que el pipeline aprendió (medianas, medias, desviaciones y coeficientes del modelo), y cualquier otra persona de la Unidad de Datos puede cargarlo con `joblib.load()` y usarlo sobre datos nuevos sin volver a correr ninguno de los notebooks de las semanas anteriores.